# Carousel Quality Eval — Sample Test Run

This notebook walks through the evaluation pipeline in three stages:

1. **Unit tests** — validate each metric function with synthetic numpy data (no Spark, no DB)
2. **Embedding smoke test** — verify `all-MiniLM-L6-v2` loads and produces sensible similarities
3. **End-to-end mini run** — run the full job against a tiny sample from real tables

Run stages 1–2 locally; stage 3 requires a Databricks cluster with Snowflake access.

## Stage 1 — Unit Tests (no Spark, no DB)

In [ ]:
import sys
# Adjust path if running outside the fabricator repo root
# sys.path.insert(0, "/path/to/fabricator")

import numpy as np
from fabricator.repository.features.cx_discovery.store_consumer_profiles.cx_profile.carousel_eval.metrics import (
    format_compliance_score,
    avg_format_compliance,
    cuisine_coverage_recall,
    intra_list_diversity,
    similarity_matrix,
    mean_max_similarity,
    semantic_recall_at_k,
    order_history_coverage_diversity,
    title_metadata_coherence,
    composite_score,
    compute_all_metrics,
    DEFAULT_THETA,
)

### Q2 — Format Compliance Score

In [ ]:
cases = [
    # (title, food_type, cuisine_type, expected_pass_fraction_approx)
    ("Korean fried chicken",
     ["fried chicken", "korean chicken", "crispy wings", "honey garlic wings",
      "boneless bites", "spicy drumsticks", "soy glazed chicken"],
     ["Korean"],
     "GOOD"),
    ("Fresh salads and bowls",          # too many words + adjective
     ["caesar salad", "greek salad"],   # food_type too short
     ["Mediterranean"],
     "BAD - word count + adjective + food_type count"),
    ("wine and cocktails",              # alcohol
     ["red wine", "white wine", "sparkling wine", "cocktail mix", "sangria blend"],
     ["Bar"],
     "BAD - alcohol"),
    ("Sushi rolls",
     ["california roll", "tuna roll", "salmon nigiri", "dragon roll", "spicy tuna",
      "rainbow roll", "shrimp tempura roll"],
     ["Japanese"],
     "GOOD"),
]

print(f"{'Title':<30} {'Score':>6}  Note")
print("-" * 70)
for title, ft, ct, note in cases:
    score = format_compliance_score(title, ft, ct)
    print(f"{title:<30} {score:>6.2f}  {note}")

### R3 — Cuisine Coverage Recall

In [ ]:
# Consumer ordered: Korean, Japanese, Thai  →  all Asian top-level family
# Carousels cover: Korean, Sichuan, Mexican
carousel_cuisines = [["Korean"], ["Sichuan"], ["Mexican"], [None], ["American"]]
order_cuisines = ["Korean", "Japanese", "Thai", "Korean", "Japanese"]  # flat list

ccr = cuisine_coverage_recall(carousel_cuisines, order_cuisines)
print(f"CCR (expect ~0.33, only Asian family matched): {ccr:.3f}")

# Perfect coverage
carousel_cuisines_perfect = [["Cantonese"], ["Japanese"], ["Thai"]]
ccr_perfect = cuisine_coverage_recall(carousel_cuisines_perfect, order_cuisines)
print(f"CCR perfect (expect 1.0): {ccr_perfect:.3f}")

### D1 — Intra-List Diversity (ILD)

In [ ]:
rng = np.random.default_rng(42)

# 10 identical vectors → ILD ≈ 0
base = rng.standard_normal(384).astype(np.float32)
base /= np.linalg.norm(base)
identical = np.stack([base] * 10)
print(f"ILD (identical carousels, expect ≈0): {intra_list_diversity(identical):.4f}")

# 10 random unit vectors → ILD ≈ 1
random_vecs = rng.standard_normal((10, 384)).astype(np.float32)
random_vecs /= np.linalg.norm(random_vecs, axis=1, keepdims=True)
print(f"ILD (random carousels, expect > 0.9): {intra_list_diversity(random_vecs):.4f}")

# Moderate diversity — 3 clusters of similar carousels
cluster_centers = rng.standard_normal((3, 384)).astype(np.float32)
cluster_centers /= np.linalg.norm(cluster_centers, axis=1, keepdims=True)
clustered = []
for i in range(10):
    v = cluster_centers[i % 3] + 0.1 * rng.standard_normal(384).astype(np.float32)
    clustered.append(v / np.linalg.norm(v))
clustered = np.stack(clustered)
print(f"ILD (3-cluster carousels, expect moderate): {intra_list_diversity(clustered):.4f}")

### R1 / R2 / D2 — Similarity-matrix metrics

In [ ]:
rng = np.random.default_rng(0)

def unit(v):
    return v / np.linalg.norm(v, axis=-1, keepdims=True)

# Scenario A: carousels perfectly match ordered items
item_embs = unit(rng.standard_normal((20, 384)).astype(np.float32))
carousel_embs_perfect = item_embs[:10].copy()  # first 10 items become carousels
sim_perfect = similarity_matrix(item_embs, carousel_embs_perfect)

print("=== Scenario A: perfect match ===")
print(f"  MMS    (expect ≈1.0):  {mean_max_similarity(sim_perfect):.4f}")
print(f"  SR@5   (expect 1.0):   {semantic_recall_at_k(sim_perfect, k=5, theta=0.45):.4f}")
print(f"  SR@10  (expect 1.0):   {semantic_recall_at_k(sim_perfect, k=10, theta=0.45):.4f}")
print(f"  OHCD   (expect 1.0):   {order_history_coverage_diversity(sim_perfect):.4f}")

# Scenario B: carousels are random (unrelated to ordered items)
carousel_embs_random = unit(rng.standard_normal((10, 384)).astype(np.float32))
sim_random = similarity_matrix(item_embs, carousel_embs_random)

print("\n=== Scenario B: random carousels ===")
print(f"  MMS    (expect low):   {mean_max_similarity(sim_random):.4f}")
print(f"  SR@5   (expect ≈0):    {semantic_recall_at_k(sim_random, k=5, theta=0.45):.4f}")
print(f"  OHCD   (expect varied): {order_history_coverage_diversity(sim_random):.4f}")

# Scenario C: all carousels are the same vector (redundant)
carousel_embs_redundant = np.stack([carousel_embs_random[0]] * 10)
sim_redundant = similarity_matrix(item_embs, carousel_embs_redundant)

print("\n=== Scenario C: all carousels identical ===")
print(f"  MMS:    {mean_max_similarity(sim_redundant):.4f}")
print(f"  OHCD   (expect 0.1 — only 1 carousel used): {order_history_coverage_diversity(sim_redundant):.4f}")

### Q1 — Title–Metadata Coherence

In [ ]:
rng = np.random.default_rng(7)

def unit(v):
    return v / np.linalg.norm(v, axis=-1, keepdims=True)

# High coherence: title_emb ≈ food_type_emb
title_embs_good = unit(rng.standard_normal((10, 384)).astype(np.float32))
food_type_embs_good = unit(
    title_embs_good + 0.05 * rng.standard_normal((10, 384)).astype(np.float32)
)
print(f"TMC (high coherence, expect > 0.9): {title_metadata_coherence(title_embs_good, food_type_embs_good):.4f}")

# Low coherence: title_emb and food_type_emb are unrelated
title_embs_bad = unit(rng.standard_normal((10, 384)).astype(np.float32))
food_type_embs_bad = unit(rng.standard_normal((10, 384)).astype(np.float32))
print(f"TMC (low coherence, expect ≈0):     {title_metadata_coherence(title_embs_bad, food_type_embs_bad):.4f}")

### Composite score

In [ ]:
# All metrics present
all_ones  = {"mms": 1.0, "sr_at_5": 1.0, "ccr": 1.0, "ild": 1.0, "ohcd": 1.0, "tmc": 1.0, "fcs": 1.0}
all_zeros = {"mms": 0.0, "sr_at_5": 0.0, "ccr": 0.0, "ild": 0.0, "ohcd": 0.0, "tmc": 0.0, "fcs": 0.0}
partial   = {"mms": 0.8, "sr_at_5": 0.7, "ccr": None, "ild": 0.6, "ohcd": None, "tmc": 0.9, "fcs": 1.0}

print(f"All 1s  → composite: {composite_score(all_ones):.3f}")
print(f"All 0s  → composite: {composite_score(all_zeros):.3f}")
print(f"Partial → composite: {composite_score(partial):.3f}  (ccr + ohcd missing, weights renormalised)")
print(f"Empty   → composite: {composite_score({})}")

### Full `compute_all_metrics` with synthetic data

In [ ]:
import json

rng = np.random.default_rng(99)

def rand_unit(shape):
    v = rng.standard_normal(shape).astype(np.float32)
    return v / np.linalg.norm(v, axis=-1, keepdims=True)

K = 10   # carousels
N = 30   # ordered items
D = 384  # embedding dim

carousel_embs  = rand_unit((K, D)).tolist()
title_embs     = rand_unit((K, D)).tolist()
food_type_embs = rand_unit((K, D)).tolist()
item_embs      = rand_unit((N, D)).tolist()

titles = [
    "Korean fried chicken", "Japanese ramen", "Spicy Thai curry",
    "Mexican street tacos", "Italian pasta", "Indian butter chicken",
    "Greek gyros", "Vietnamese pho", "Sushi rolls", "Pad Thai noodles",
]
food_types = [
    ["fried chicken", "korean chicken", "crispy wings", "honey garlic wings", "spicy drumsticks"],
] * K  # simplified
cuisine_types = [["Korean"], ["Japanese"], ["Thai"], ["Mexican"], ["Italian"],
                 ["Northern Indian"], ["Greek"], ["Vietnamese"], ["Japanese"], ["Thai"]]
item_cuisines  = [["Korean"], ["Japanese"], ["Thai"]] * 10  # 30 items

result = compute_all_metrics(
    carousel_ranks=list(range(K)),
    titles=titles,
    food_types=food_types,
    cuisine_types=cuisine_types,
    carousel_emb_list=carousel_embs,
    title_emb_list=title_embs,
    food_type_emb_list=food_type_embs,
    item_emb_list=item_embs,
    item_cuisine_tag_list=item_cuisines,
)

print(json.dumps({k: round(v, 4) if v is not None else None for k, v in result.items()}, indent=2))

---
## Stage 2 — Embedding Smoke Test

Requires `sentence-transformers` installed: `pip install sentence-transformers`

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

# Carousel texts to embed
carousel_texts = [
    "Korean fried chicken: fried chicken, korean chicken, crispy wings, honey garlic wings, spicy drumsticks",
    "Japanese ramen: tonkotsu broth, shoyu ramen, miso ramen, pork belly, soft boiled egg",
    "Spicy Thai curry: green curry, red curry, panang curry, coconut milk curry, massaman curry",
    "Mexican street tacos: carnitas taco, carne asada taco, al pastor taco, fish taco, birria taco",
    "Italian pasta: spaghetti carbonara, penne arrabbiata, fettuccine alfredo, rigatoni bolognese, cacio e pepe",
]

# Order history items
item_texts = [
    "Spicy Chicken Sandwich. Crispy fried chicken with gochujang sauce on a brioche bun.",
    "Tonkotsu Ramen. Rich pork bone broth with chashu pork, menma, and nori.",
    "Green Curry. Coconut milk based green curry with vegetables and jasmine rice.",
    "Caesar Salad. Romaine lettuce, parmesan, croutons, caesar dressing.",  # off-topic
    "Carne Asada Burrito. Grilled beef, rice, beans, pico de gallo, guacamole.",
]

carousel_embs = model.encode(carousel_texts, normalize_embeddings=True)
item_embs = model.encode(item_texts, normalize_embeddings=True)

sim = item_embs @ carousel_embs.T

print("Cosine similarity matrix (items × carousels):")
print("Items (rows) vs Carousels (cols): Korean, Japanese, Thai, Mexican, Italian")
print()
for i, item_text in enumerate(item_texts):
    label = item_text[:40]
    row = "  ".join(f"{s:5.2f}" for s in sim[i])
    best_j = sim[i].argmax()
    print(f"{label:<42} [{row}]  → best: carousel {best_j}")

In [ ]:
# Verify metric values on this realistic example
print(f"MMS:    {mean_max_similarity(sim):.4f}  (expect high for first 3 items, lower for caesar)")
print(f"SR@5:   {semantic_recall_at_k(sim, k=5, theta=0.35):.4f}")
print(f"ILD:    {intra_list_diversity(carousel_embs):.4f}  (5 diverse carousels, expect > 0.4)")
print(f"OHCD:   {order_history_coverage_diversity(sim):.4f}")

In [ ]:
# Title vs food_type coherence on realistic examples
titles_only = [
    "Korean fried chicken",
    "Japanese ramen",
    "Spicy Thai curry",
    "Mexican street tacos",
    "Italian pasta",
]
food_type_strs = [
    "fried chicken, korean chicken, crispy wings, honey garlic wings, spicy drumsticks",
    "tonkotsu broth, shoyu ramen, miso ramen, pork belly, soft boiled egg",
    "green curry, red curry, panang curry, coconut milk curry, massaman curry",
    "carnitas taco, carne asada taco, al pastor taco, fish taco, birria taco",
    "spaghetti carbonara, penne arrabbiata, fettuccine alfredo, rigatoni bolognese, cacio e pepe",
]
mismatched_food_type_strs = [
    # Titles unchanged but food_types shuffled to mismatch
    "spaghetti carbonara, penne arrabbiata, fettuccine alfredo, rigatoni bolognese, cacio e pepe",
    "fried chicken, korean chicken, crispy wings, honey garlic wings, spicy drumsticks",
    "tonkotsu broth, shoyu ramen, miso ramen, pork belly, soft boiled egg",
    "green curry, red curry, panang curry, coconut milk curry, massaman curry",
    "carnitas taco, carne asada taco, al pastor taco, fish taco, birria taco",
]

title_embs_real  = model.encode(titles_only, normalize_embeddings=True)
ft_embs_matched  = model.encode(food_type_strs, normalize_embeddings=True)
ft_embs_mismatch = model.encode(mismatched_food_type_strs, normalize_embeddings=True)

tmc_matched  = title_metadata_coherence(title_embs_real, ft_embs_matched)
tmc_mismatch = title_metadata_coherence(title_embs_real, ft_embs_mismatch)

print(f"TMC matched food_types  (expect high): {tmc_matched:.4f}")
print(f"TMC shuffled food_types (expect low):  {tmc_mismatch:.4f}")

---
## Stage 3 — End-to-End Mini Run (Databricks + Snowflake)

> Run this stage on a Databricks cluster with the `fabricator` package and Snowflake connector installed.
> Set `SAMPLE_CONSUMERS` to a small number to keep runtime short.

In [ ]:
SAMPLE_CONSUMERS = 500   # reduce to 100 for a quick smoke test
ACTIVE_DATE      = "2026-03-22"  # change to a date with generated carousels
CAROUSEL_TABLE   = "proddb.ml.cx_profile_generated_carousels_ebr"
ORDER_HISTORY_LOOKBACK_DAYS = 90

In [ ]:
import datetime
from fabricator_core.connectors.snowflake import load_data_spark
from fabricator_core.connectors.context_io import load_from_context
from fabricator_core.core.contexts.dataset_context import DatasetContext
from pyspark.sql import functions as F
from pyspark.sql import types as T

from fabricator.repository.features.cx_discovery.store_consumer_profiles.cx_profile.carousel_eval.eval_job import (
    embed_texts_udf,
    _compute_group,
    _METRICS_SCHEMA,
    EMBEDDING_MODEL,
)

start_date = (
    datetime.datetime.strptime(ACTIVE_DATE, "%Y-%m-%d")
    - datetime.timedelta(days=ORDER_HISTORY_LOOKBACK_DAYS)
).date().isoformat()

In [ ]:
# ── Load a small sample of consumers that have carousels ──────────────────
df_sample_consumers = load_data_spark(
    f"""
    SELECT DISTINCT consumer_id
    FROM {CAROUSEL_TABLE}
    WHERE active_date = '{ACTIVE_DATE}'
    LIMIT {SAMPLE_CONSUMERS}
    """
)

df_carousels = load_data_spark(
    f"""
    SELECT
        consumer_id,
        day_part,
        carousel_rank,
        title,
        tags['food_type']    AS food_type,
        tags['cuisine_type'] AS cuisine_type
    FROM {CAROUSEL_TABLE}
    WHERE active_date = '{ACTIVE_DATE}'
    """
).join(df_sample_consumers, on="consumer_id", how="inner")

print(f"Carousel rows: {df_carousels.count()}")
df_carousels.show(5, truncate=80)

In [ ]:
# ── Load order history for the same consumers ─────────────────────────────
df_items_raw = load_from_context(
    DatasetContext.from_source("purchased_items_for_profiles_base_instance"),
    active_date_range=f"{start_date}...{ACTIVE_DATE}",
    allowed_missing_dates=2,
)

df_items = (
    df_items_raw
    .join(df_sample_consumers, on="consumer_id", how="inner")
    .filter(F.col("is_cng") == 0)
    .filter(F.col("is_group_order") == 0)
    .select(
        "consumer_id",
        F.col("daypart_breakdown").alias("day_part"),
        "item_id",
        "item_name",
        F.coalesce(F.col("description"), F.lit("")).alias("description"),
        F.split(F.coalesce(F.col("cuisine_tags_from_menu"), F.lit("")), ", ").alias("cuisine_tags"),
    )
    .dropDuplicates(["consumer_id", "day_part", "item_id"])
)

print(f"Item rows: {df_items.count()}")
df_items.show(5, truncate=80)

In [ ]:
# ── Embed carousels ───────────────────────────────────────────────────────
df_carousels = (
    df_carousels
    .withColumn("food_type_str", F.concat_ws(", ", F.col("food_type")))
    .withColumn("carousel_text", F.concat(F.col("title"), F.lit(": "), F.col("food_type_str")))
    .withColumn("carousel_emb", embed_texts_udf(F.col("carousel_text")))
    .withColumn("title_emb", embed_texts_udf(F.col("title")))
    .withColumn("food_type_emb", embed_texts_udf(F.col("food_type_str")))
)
print("Carousel embeddings added.")

In [ ]:
# ── Embed items ───────────────────────────────────────────────────────────
df_items = (
    df_items.withColumn(
        "item_text",
        F.when(F.col("description") != "",
               F.concat(F.col("item_name"), F.lit(". "), F.col("description")))
        .otherwise(F.col("item_name"))
    )
    .withColumn("item_emb", embed_texts_udf(F.col("item_text")))
)
print("Item embeddings added.")

In [ ]:
# ── Union into combined DataFrame and run applyInPandas ───────────────────
_null_float_array = F.lit(None).cast(T.ArrayType(T.FloatType()))
_null_str_array   = F.lit(None).cast(T.ArrayType(T.StringType()))
_null_str         = F.lit(None).cast(T.StringType())
_null_int         = F.lit(None).cast(T.IntegerType())

df_carousel_rows = df_carousels.select(
    "consumer_id", "day_part",
    F.lit("carousel").alias("row_type"),
    F.col("carousel_rank").cast(T.IntegerType()),
    "title", "food_type", "cuisine_type",
    "carousel_emb", "title_emb", "food_type_emb",
    _null_float_array.alias("item_emb"),
    _null_str_array.alias("item_cuisine_tags"),
)

df_item_rows = df_items.select(
    "consumer_id", "day_part",
    F.lit("item").alias("row_type"),
    _null_int.alias("carousel_rank"),
    _null_str.alias("title"),
    _null_str_array.alias("food_type"),
    _null_str_array.alias("cuisine_type"),
    _null_float_array.alias("carousel_emb"),
    _null_float_array.alias("title_emb"),
    _null_float_array.alias("food_type_emb"),
    "item_emb",
    F.col("cuisine_tags").alias("item_cuisine_tags"),
)

df_combined = df_carousel_rows.unionByName(df_item_rows)

df_eval = (
    df_combined
    .groupBy("consumer_id", "day_part")
    .applyInPandas(_compute_group, schema=_METRICS_SCHEMA)
    .withColumn("active_date", F.lit(ACTIVE_DATE))
    .withColumn("embedding_model", F.lit(EMBEDDING_MODEL))
    .cache()
)

print(f"Eval rows: {df_eval.count()}")
df_eval.show(10, truncate=True)

In [ ]:
# ── Summary statistics ────────────────────────────────────────────────────
metrics = ["mms", "sr_at_3", "sr_at_5", "sr_at_10", "ccr", "ild", "ohcd", "tmc", "fcs", "composite_quality_score"]

df_eval.select(
    *[F.round(F.avg(m), 4).alias(f"{m}_mean") for m in metrics]
).show(truncate=False)

In [ ]:
# ── Per-daypart breakdown ─────────────────────────────────────────────────
df_eval.groupBy("day_part").agg(
    F.count("*").alias("n"),
    F.round(F.avg("mms"), 4).alias("mms_mean"),
    F.round(F.avg("sr_at_5"), 4).alias("sr_at_5_mean"),
    F.round(F.avg("ild"), 4).alias("ild_mean"),
    F.round(F.avg("composite_quality_score"), 4).alias("composite_mean"),
).orderBy("day_part").show(truncate=False)

In [ ]:
# ── Low-quality consumers (composite < 0.4) ───────────────────────────────
df_low = (
    df_eval
    .filter(F.col("composite_quality_score") < 0.4)
    .orderBy(F.col("composite_quality_score").asc())
    .limit(20)
)
print(f"Low-quality rows (composite < 0.4): {df_low.count()}")
df_low.show(truncate=True)